# LLM Environment Validation

## Objective

Validate the available computational environments for the MSc dissertation.

## Environment

- Computational Teaching JupyterHub
- NVIDIA A40 (46 GB VRAM)
- 503 GB RAM
- 48 CPU cores
- PyTorch 2.3.1+cu121

## Models Tested

| Model | Status |
|-------|--------|
| sshleifer/tiny-gpt2 | ✅ Success |
| Qwen2.5-1.5B-Instruct | ✅ Success |
| Llama-3.2-1B-Instruct | ✅ Success |
| Llama-3.3-70B-Instruct | ⚠️ Download successful; loading failed due to GPU memory limitations |

## Conclusions

- The environment is suitable for small and medium LLMs.
- Llama 70B downloads successfully but cannot be loaded entirely on a single NVIDIA A40 (46 GB VRAM) using the tested 4-bit configuration.
- Additional GPU resources or a different deployment strategy would be required for Llama 70B/Centaur.

In [1]:
# tiny-gpt2

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

inputs = tokenizer("Hello, my name is", return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA A40
Hello, my name is vendors antibiotic substatisf trilogy antibiotic Rh004 Motorola Habitreement Hancock subst circumcised Hancockting confir confir Observpress


In [3]:
# Llama 3.2 1B

In [4]:
from huggingface_hub import login

login("HUGGINGFACE_TOKEN_REDACTED")

In [5]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '69f37fae1beca22174c28db0', 'name': 'oscargallegos', 'fullname': 'Oscar Alejandro Gallegos Villarreal', 'email': 'oscar5198@hotmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': '/avatars/5bd7575391396f5c9aaff8cd00c49f22.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'llama-dissertation', 'role': 'read', 'createdAt': '2026-06-23T14:09:10.824Z'}}}


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

messages = [
    {"role": "user", "content": "Explain what reverb does in music production in simple terms."}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

PyTorch: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA A40


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

Cutting Knowledge Date: December 2023
Today Date: 23 Jun 2026

user

Explain what reverb does in music production in simple terms.assistant

Reverb is a powerful effect in music production that can add depth, width, and atmosphere to your sound.

Imagine you're in a large concert hall or a quiet room. The acoustics of the space are what make it sound big and spacious, right? That's what reverb does for your music.

Reverb is a digital effect that mimics the way sound behaves when it travels through a physical space, like a room or a hall. It simulates the way sound waves bounce off surfaces, creating a sense of space and distance.

When you apply reverb to a sound, it


In [7]:
# Qwen 2.5

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

messages = [
    {
        "role": "user",
        "content": "Explain what EQ does in music production."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain what EQ does in music production.
assistant
EQ (Equalization) is an essential tool in the world of music production and sound engineering. It stands for Equalizer or Equalisation, depending on the region you're from. In essence, EQ is used to adjust the balance between different frequencies within an audio signal.

Here's how it works:

1. **Frequency Range Adjustment**: EQ allows producers and engineers to manipulate the frequency content of a sound. This means they can boost or cut certain ranges of frequencies while reducing others.

2. **Coloring Sounds**: By adjusting the levels of specific frequencies, producers can create unique sounds that match their artistic


In [10]:
# Llama 3.3 70B in 4-bit

In [11]:
import torch
import transformers
import tokenizers
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.3.1+cu121
Transformers: 4.46.3
Tokenizers: 0.20.3
BitsAndBytes: 0.44.1
CUDA: True
GPU: NVIDIA A40


In [12]:
import os
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_HUB_CACHE"] = "/tmp/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_cache/datasets"

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

cache_dir = "/tmp/hf_cache"
model_name = "meta-llama/Llama-3.3-70B-Instruct"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    cache_dir=cache_dir,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    cache_dir=cache_dir,
)

messages = [
    {"role": "user", "content": "Explain compression in music production in simple terms."}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

PyTorch: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA A40


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# Centaur